# Info

This notebook downloads raw data from a Google Firebase storage bucket.
The bucket stores:
- audio/transcript data generated by the "chiara-speech2text_frontend" app
- audio/transcript data generated through Whatsapp voice messages
- audio/transcript data generated through a custom keyboard

# ToDo

- Look for processed folders inside the 'folders.json' at the firebase root (they have: "is_processed": true)
- Download them automatically
- Store their (local) folder paths inside a file that can be used by the next notebook  

# Set parameters

In [ ]:
# Credentials

# env file with configs and credentials
ENV_FILE = "/home/lucar/Development/chiara-speech2text_private-v2/envs/.env"

In [ ]:
# Directories

# Output dataset directory
DIR_DATA_RAW = "/home/lucar/Development/chiara-speech2text_data/data_training/data_raw"

In [ ]:
# Folders for download
# NOTE: Folders name defined here shall match those defined in the Firebase bucket

folders = [
    # whatsapp data
    '202311_chiara_whatsapp',
    '202312_chiara_whatsapp',
    '202401_chiara_whatsapp',
    '202402_chiara_whatsapp',
    '202403_chiara_whatsapp',
    '202404_chiara_whatsapp',
    '202405_chiara_whatsapp',
    '202406_chiara_whatsapp',
    '202407_chiara_whatsapp',
    '202408_chiara_whatsapp',
    '202409_chiara_whatsapp',
    # frontend data
    '20251020_chiara_frontend',
    '20251021_chiara_frontend',
    '20251022_chiara_frontend',
    '20251023_chiara_frontend',
    '20251024_chiara_frontend',
    '20251029_chiara_frontend',
    '20251110_chiara_frontend',
    '20251123_chiara_frontend',
    '20260104_chiara_frontend',
    '20260105_chiara_frontend',
    # dgx (keyboard) data
    '20260117_dgx',
    '20260118_dgx',
    '20260119_dgx',
    '20260121_dgx',
    '20260122_dgx',
    '20260123_dgx',
    '20260124_dgx',
    '20260125_dgx',
    '20260126_dgx',
    '20260127_dgx',
    '20260128_dgx',
    '20260129_dgx',
    '20260130_dgx',
    '20260131_dgx',
    '20260201_dgx',
    '20260202_dgx',
    '20260203_dgx',
    '20260206_dgx',
]

# Initialize notebook

In [ ]:
# Install packages

%pip install --quiet gsutil

In [ ]:
# Delete old raw data folder

# DIR_DATA_RAW
!rm -rf {DIR_DATA_RAW}
!mkdir  {DIR_DATA_RAW}

In [ ]:
# Import variables from .env file

# Load the variables from the .env file into the environment
from dotenv import load_dotenv
load_dotenv(ENV_FILE)

In [ ]:
# Disable ipywidgets bars (they cause visualization errors with transforms library in notebooks)

import os
os.environ["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

import datasets
datasets.disable_progress_bars()

In [ ]:
# Download folders

# Access variables using os.environ.get('VARIABLE_NAME')
import os
DIR_FIREBASE_BUCKET = os.environ.get("TRAINER_FIREBASE_BUCKET")

# Print
print(f"I will download data from the firebase bucket: {DIR_FIREBASE_BUCKET}")

# Gather all unique folders to download
folders_to_download = set()

if folders:
    folders_to_download.update(folders)

print(f"Found {len(folders_to_download)} unique folders to download.")
print(f"Source Bucket:  {DIR_FIREBASE_BUCKET}")
print(f"Destination:    {DIR_DATA_RAW}\n")

# Ensure destination directory exists
os.makedirs(DIR_DATA_RAW, exist_ok=True)

# Download each folder using gcloud storage
for folder in folders_to_download:
    print(f"-> Downloading '{folder}'...")
    
    # We use gcloud storage cp instead of gsutil
    !gcloud storage cp -r gs://{DIR_FIREBASE_BUCKET}/{folder} {DIR_DATA_RAW}/
    
print("\n✓ All downloads completed.")

In [ ]:
# cleanup
try:
    del folders_to_download
    del folder
except:
    pass

# Inspect raw data

In [ ]:
# Inspect a user-defined file

audio_file = "20251020_chiara_frontend/36.wav"

#
import os
from IPython.display import Audio
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd
%matplotlib inline
import librosa.display
import matplotlib.pyplot as plt



# --- Widget to listen to audio
audio_path = os.path.join(DIR_DATA_RAW, audio_file)
if os.path.exists(audio_path):
    display(Audio(filename=audio_path))
else:
    print(f"Audio file '{audio_file}' not found)")



# get audio data
audio, sample_rate = librosa.load(audio_path)
ipd.Audio(audio_path, rate=sample_rate)



# --- Plot waveform
plt.figure()
plt.rcParams['figure.figsize'] = (15,7)
plt.title('Waveform of Audio Example')
plt.ylabel('Amplitude')

_ = librosa.display.waveshow(audio, color='blue')
plt.show()  # Add this to display the waveform



# --- Plot spectrogram
plt.figure()
spec = np.abs(librosa.stft(audio))
spec_db = librosa.amplitude_to_db(spec, ref=np.max)  # Decibels

# Use log scale to view frequencies
librosa.display.specshow(spec_db, y_axis='log', x_axis='time')
plt.colorbar()
plt.title('Audio Spectrogram');



# --- Mel spectrogram
plt.figure() 
mel_spec = librosa.feature.melspectrogram(y=audio, sr=sample_rate)
mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

librosa.display.specshow(
    mel_spec_db, x_axis='time', y_axis='mel')
plt.colorbar()
plt.title('Mel Spectrogram')

In [ ]:
# cleanup
try:
    del mel_spec, mel_spec_db
    del audio_file, audio_path, audio, sample_rate, spec, spec_db
except:
    pass

In [ ]:
# Inspect a user-defined folder


# Show unique strings within selected folder
folder = "20251020_chiara_frontend"

import os
from collections import defaultdict
import json

def get_unique_transcripts(folder_path):
    unique_transcripts = set()
    
    # Walk through the specified folder
    for root, dirs, files in os.walk(folder_path):
        if 'data.json' in files:
            file_path = os.path.join(root, 'data.json')
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    content = json.load(f)
                    
                    # Assuming your json structure has a 'data' array based on standard formatting
                    if 'data' in content and isinstance(content['data'], list):
                        for entry in content['data']:
                            if 'reviewer_corrected_transcript' in entry:
                                transcript = entry['reviewer_corrected_transcript']
                                # You can add '.strip()' or '.lower()' here if you want to normalize the text
                                unique_transcripts.add(transcript)
            except Exception as e:
                print(f"Error reading {file_path}: {e}")
                
    return sorted(list(unique_transcripts))

# Get all available folders
if os.path.exists(DIR_DATA_RAW):
    all_folders = sorted([item for item in os.listdir(DIR_DATA_RAW) 
                         if os.path.isdir(os.path.join(DIR_DATA_RAW, item))])
    
    if not all_folders:
        print(f"Error: No folders found in {DIR_DATA_RAW}")
    
    else:
        # Display available folders
        print(f"\nAvailable folders ({len(all_folders)}):")
        for idx, folder in enumerate(all_folders, 1):
            print(f"  {idx}. {folder}")
              
        try:
            if folder in all_folders:
                selected_folder = folder
                print(f"Selected folder: {selected_folder}")
            else:
                raise ValueError(f"Folder '{folder}' not found. Available folders: {all_folders}")
        except ValueError as e:
            print(f"Error: {e}")
            raise
        
        # Process selected folder
        selected_folder_path = os.path.join(DIR_DATA_RAW, selected_folder)
        print("path: " + selected_folder_path)

        unique_items = get_unique_transcripts(selected_folder_path)
        print(f"Found {len(unique_items)} unique instances:\n")
        for item in unique_items:
            print(f"- {item}")

else:
    print(f"Error: DIR_DATA_RAW '{DIR_DATA_RAW}' does not exist")

# cleanup
try:
    del all_folders, folder, selected_folder
    del selected_folder_path, txt_files, txt_file, txt_path
    del content, unique_strings, sorted_strings, idx, string
    del f
    del unique_items
except:
    pass

In [ ]:
# cleanup
try:
    del idx, item, unique_items
except:
    pass

# Next steps

See notebook `2-prepare-dataset.ipynb`